In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import cv2
import os
from ultralytics import YOLO

In [2]:
output_folder = '/content/drive/MyDrive/City_Fire_Proj/dataset/test_results'

In [ ]:
if not os.path.exists(output_folder):
  os.makedirs(output_folder)

In [ ]:
video_path = 'fresh_video'
cap = cv2.VideoCapture(video_path)

In [ ]:
out_video_path = os.path.join(output_folder,video_path)

In [ ]:
model = YOLO('/content/runs/detect/City_Fire_Proj/smoke_fire_v1/weights/best.pt')

In [ ]:
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
hight = int(cap.get(cv2.CAP_PROP_FRAME_HIGHT))
fps = int(cap.get(cv2.CAP_PROP_FRAME_FPS))

In [ ]:
out = cv2.VideoWriter(out_video_path, cv2.VideoWriter_fourcc(*'mp4v'),fps,(width,hight))

In [ ]:
frame_count = 0
skip_frame = 3
while cap.isOpened():
  ret,frame = cap.read()
  if not ret:
    raise RuntimeError('Ошибка: кадр не прочтён')
  frame_count += 1
  if frame_count % skip_frame != 0:
    out.write(frame)
    continue
  result = model(frame,conf=0.5)[0]
  if len(result.boxs) > 0:
    for box in result.boxs:
      class_id = int(box.cls[0])
      name_cls = model.name(class_id)
      print(f'!!! Discovered {name_cls.upper()}!!!')
  annotated_frame = result.plot()
  out.write(annotated_frame)

out.release()
cap.release()
print(f'Видео сохранено в папке: {out_video_path}')